# Simple CNN Walkthrough (MNIST digits)
Run each cell from top to bottom. Every step prints or plots something so you can see what is happening.

Install first if needed: `pip install torch torchvision matplotlib`

## Step 0: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

## Step 1: Data
Each image is **28x28 pixels**, grayscale (1 channel), showing a digit 0-9.
`ToTensor` converts the picture into numbers between 0 and 1.
The `DataLoader` feeds images to the model in small groups called **batches**.

In [ ]:
transform = transforms.ToTensor()
train_data = datasets.MNIST("data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

print("Training images:", len(train_data))
print("Test images:", len(test_data))

### Look at some images
This is what the computer sees: just a grid of numbers. Label is shown on top.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for i, ax in enumerate(axes):
    image, label = train_data[i]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.show()

image, label = train_data[0]
print("Shape of one image:", image.shape, " -> [channels, height, width]")

## Step 2: The layers, one at a time
Before building the full model, let's try each layer on one image and watch the shape change.

### 2a. Convolution
Small filters (3x3) slide over the image and detect patterns like edges.
We use **8 filters**, so we get **8 feature maps** out. `padding=1` keeps the size at 28x28.

In [ ]:
conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

x = image.unsqueeze(0)          # add a batch dimension -> [1, 1, 28, 28]
print("Input shape:      ", x.shape)

after_conv = conv1(x)
print("After convolution:", after_conv.shape, " -> 8 feature maps")

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(after_conv[0, i].detach(), cmap="gray")
    ax.set_title(f"Filter {i+1}")
    ax.axis("off")
plt.show()

### 2b. ReLU (activation)
Turns every negative number into 0 and keeps positives.
This adds non-linearity so the network can learn more than straight-line patterns.

In [ ]:
relu = nn.ReLU()
after_relu = relu(after_conv)

print("Before ReLU, min value:", after_conv.min().item())
print("After ReLU,  min value:", after_relu.min().item())
print("Shape unchanged:", after_relu.shape)

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(after_relu[0, i].detach(), cmap="gray")
    ax.axis("off")
plt.show()

### 2c. Pooling (Max Pooling)
Shrinks each feature map by half. In every 2x2 block it keeps only the largest value.
Result: less data, faster, and more tolerant to small shifts in the image.

In [ ]:
pool = nn.MaxPool2d(kernel_size=2)
after_pool = pool(after_relu)

print("Before pooling:", after_relu.shape)
print("After pooling: ", after_pool.shape, " -> height and width cut in half")

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(after_pool[0, i].detach(), cmap="gray")
    ax.axis("off")
plt.show()

### 2d. Flatten
Stretches the 3D feature maps into one long list so the final layer can read them.

In [ ]:
flat = after_pool.view(1, -1)
print("Before flatten:", after_pool.shape)
print("After flatten: ", flat.shape, "-> 8 x 14 x 14 =", 8 * 14 * 14)

### 2e. Fully connected layer (output)
Takes the flat list and produces **10 scores**, one for each digit 0-9.
The highest score is the model's prediction. (Untrained, so the guess is random for now.)

In [ ]:
fc = nn.Linear(8 * 14 * 14, 10)
scores = fc(flat)
print("Output scores:", scores.detach().numpy().round(2))
print("Predicted digit (untrained):", scores.argmax().item())

## Step 3: Put it all together in one model
Same layers as above, now with **two** convolution blocks so the model learns
simple patterns first (edges), then combined patterns (curves, shapes).

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)    # 1 -> 8 feature maps
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)   # 8 -> 16 feature maps
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16 * 7 * 7, 10)                       # 784 -> 10 scores

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # [batch, 8, 14, 14]
        x = self.pool(self.relu(self.conv2(x)))   # [batch, 16, 7, 7]
        x = x.view(x.size(0), -1)                 # [batch, 784]
        x = self.fc(x)                            # [batch, 10]
        return x

model = SimpleCNN()
print(model)

## Step 4: Loss function and optimizer
- **Loss**: measures how wrong the model is. `CrossEntropyLoss` also applies softmax internally (turns scores into probabilities).
- **Optimizer**: updates the filters and weights to reduce the loss.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Step 5: Training loop
For each batch of images:
1. **Forward pass**: make a guess
2. **Loss**: measure how wrong it is
3. **Backpropagation**: work out what to change
4. **Optimizer step**: update the weights

One **epoch** = one full pass over all training images.

In [ ]:
epochs = 3
loss_history = []

for epoch in range(epochs):
    model.train()
    for images, labels in train_loader:
        predictions = model(images)             # 1. forward pass
        loss = loss_fn(predictions, labels)     # 2. how wrong?

        optimizer.zero_grad()                   # clear old gradients
        loss.backward()                         # 3. backpropagation
        optimizer.step()                        # 4. update weights

        loss_history.append(loss.item())

    print(f"Epoch {epoch + 1}/{epochs} done - last batch loss: {loss.item():.4f}")

### Did it learn?
The loss should go down over time. Lower loss = fewer mistakes.

In [ ]:
plt.plot(loss_history)
plt.xlabel("Batch number")
plt.ylabel("Loss")
plt.title("Training loss")
plt.show()

## Step 6: Testing
Check accuracy on images the model has **never seen** during training.

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():                           # no learning, just evaluating
    for images, labels in test_loader:
        outputs = model(images)
        predicted = outputs.argmax(dim=1)       # digit with the highest score
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

print(f"Test accuracy: {100 * correct / total:.2f}%")

## Step 7: See some predictions
Green = correct, red = wrong.

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    preds = model(images).argmax(dim=1)

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].squeeze(), cmap="gray")
    color = "green" if preds[i] == labels[i] else "red"
    ax.set_title(f"Pred: {preds[i].item()}  (True: {labels[i].item()})", color=color)
    ax.axis("off")
plt.show()

## Recap
1. **Convolution** finds patterns with small filters
2. **ReLU** adds non-linearity
3. **Pooling** shrinks and keeps the strongest signals
4. **Flatten** turns feature maps into a list
5. **Fully connected layer** produces a score per class
6. **Loss + optimizer + backpropagation** teach the model